# Student Info Agent using LangChain and Gemini

In [ ]:
!pip install -q langchain langchain-google-genai langchain-core

In [ ]:
import sqlite3
import os
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"

## Create Database

In [ ]:
DB_NAME = "students.db"

if os.path.exists(DB_NAME):
    os.remove(DB_NAME)

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE students (
        student_id TEXT PRIMARY KEY,
        name TEXT,
        department TEXT,
        python INTEGER,
        database INTEGER,
        ai INTEGER,
        web INTEGER
    )
""")

students = [
    ("22CS045", "Dhanushya", "Computer Science", 85, 72, 90, 78),
    ("22CS046", "Rahul", "Computer Science", 65, 70, 68, 72),
    ("22CS047", "Priya", "Information Technology", 92, 88, 95, 90),
    ("22CS048", "Arun", "Information Technology", 55, 60, 58, 62),
    ("22CS049", "Meena", "Computer Science", 78, 85, 80, 88),
]

cursor.executemany("INSERT INTO students VALUES (?, ?, ?, ?, ?, ?, ?)", students)
conn.commit()
conn.close()

## Define Tools

In [ ]:
@tool
def get_student_info(student_id: str) -> str:
    """Returns the name and department of a student given their student_id."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, department FROM students WHERE student_id = ?",
        (student_id,),
    )
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return f"No student found with ID {student_id}"
    name, department = row
    return f"Name: {name}, Department: {department}"

In [ ]:
@tool
def get_student_marks(student_id: str) -> str:
    """Returns the python, database, ai, and web marks of a student given their student_id."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute(
        "SELECT python, database, ai, web FROM students WHERE student_id = ?",
        (student_id,),
    )
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return f"No student found with ID {student_id}"
    python, database, ai, web = row
    return f"Python: {python}, Database: {database}, AI: {ai}, Web: {web}"

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression such as sums or averages of marks and returns the result."""
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

In [ ]:
@tool
def get_passing_rules() -> str:
    """Returns the university passing rules: minimum overall average and minimum mark per subject."""
    return "Minimum overall average required: 40%. Minimum mark required in each subject: 35%."

## Create Agent

In [ ]:
tools = [get_student_info, get_student_marks, calculator, get_passing_rules]

llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that answers questions about students using the available tools. Use tools whenever needed and chain multiple tools if required to fully answer the question."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## Helper Function

In [ ]:
def ask(question: str):
    response = agent_executor.invoke({"input": question})
    print(response["output"])

## Question 1

In [ ]:
ask("What is the name and department of student 22CS045?")

## Question 2

In [ ]:
ask("What are the marks of 22CS047?")

## Question 3

In [ ]:
ask("What is the total and average mark of 22CS045?")

## Question 4

In [ ]:
ask("Is 22CS045 eligible to pass according to the university rules?")

## Challenge Question

In [ ]:
ask("I am 22CS045. Tell me my name, department, total marks, average marks, and whether I satisfy the university passing requirements.")